In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document,Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import openai
import fastembed

In [12]:
qdrant_client = QdrantClient("http://localhost:6333")

In [13]:
qdrant_client.create_collection(
    collection_name="Amazon_Electronics_Products",
    vectors_config={
        "text-embedding-3-small": VectorParams(
            size=1536,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

In [15]:
qdrant_client.create_payload_index(
    collection_name="Amazon_Electronics_Products",
    field_name="store",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [65]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


In [16]:
def get_embedding(text, model= "text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [17]:
def get_embeddings_batch(text_list, model= "text-embedding-3-small", batch_size=100):
    if(len(text_list) <= batch_size):
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed batch {counter} / {len(text_list) // batch_size + 1}")
        counter += 1
    return all_embeddings

In [ ]:
results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
    )

In [20]:
df = pd.read_json('../../data/Data_With_Images.jsonl', lines=True)
print(df.head())

               main_category  \
0            All Electronics   
1            All Electronics   
2                  Computers   
3             AMAZON FASHION   
4  Cell Phones & Accessories   

                                               title  average_rating  \
0             FS-1051 FATSHARK TELEPORTER V3 HEADSET             3.5   
1                      Ce-H22B12-S1 4Kx2K Hdmi 4Port             5.0   
2  Digi-Tatoo Decal Skin Compatible With MacBook ...             4.5   
3  NotoCity Compatible with Vivoactive 4 band 22m...             4.5   
4             Motorola Droid X Essentials Combo Pack             3.8   

   rating_number                                           features  \
0              6                                                 []   
1              1             [UPC: 662774021904, Weight: 0.600 lbs]   
2            246  [WARNING: Please IDENTIFY MODEL NUMBER on the ...   
3            233  [☛NotoCity 22mm band is designed for Vivoactiv...   
4             64  [

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2026 entries, 0 to 2025
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   main_category   2000 non-null   object 
 1   title           2026 non-null   object 
 2   average_rating  2026 non-null   float64
 3   rating_number   2026 non-null   int64  
 4   features        2026 non-null   object 
 5   description     2026 non-null   object 
 6   price           2026 non-null   float64
 7   images          2026 non-null   object 
 8   videos          2026 non-null   object 
 9   store           2026 non-null   object 
 10  categories      2026 non-null   object 
 11  details         2026 non-null   object 
 12  parent_asin     2026 non-null   object 
 13  product_id      2026 non-null   object 
dtypes: float64(2), int64(1), object(11)
memory usage: 221.7+ KB


In [22]:
len(df)

2026

In [219]:
def preprocess_description(row):
    return f"{row['title']} {row['description']} {row['main_category']} {row['features']} {row['price']} {row['categories']}"
    # return f"{row['categories']}"

In [220]:
def extract_first_large_images(row):
    return row['images'][0].get('large', None)

In [221]:
df['processed_description'] = df.apply(preprocess_description, axis=1)
df['image_url'] = df.apply(extract_first_large_images, axis=1)

In [222]:
df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,product_id,processed_description,image_url
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,96.551146,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,"[Electronics, Television & Video, Video Glasses]","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M,b86c3534-084b-449b-889e-50bd268ff964,FS-1051 FATSHARK TELEPORTER V3 HEADSET ['Telep...,https://m.media-amazon.com/images/I/41qrX56lsY...
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],96.551146,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE,945a8ca0-282f-424a-83db-646304822e23,Ce-H22B12-S1 4Kx2K Hdmi 4Port ['HDMI In - HDMI...,https://m.media-amazon.com/images/I/31OIMoOW70...
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.990000,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'AL 2Sides Video', 'url': 'https://...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble...",B07SM135LS,c67b6305-54b3-4a14-816a-532c3d2a108c,Digi-Tatoo Decal Skin Compatible With MacBook ...,https://m.media-amazon.com/images/I/31t4bj9t88...
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.990000,[{'thumb': 'https://m.media-amazon.com/images/...,[],NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{'Date First Available': 'May 29, 2020', 'Manu...",B089CNGZCW,5f511624-c19c-401e-8669-2d22afaa50dc,NotoCity Compatible with Vivoactive 4 band 22m...,https://m.media-amazon.com/images/I/41j56fjX6S...
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.990000,[{'thumb': 'https://m.media-amazon.com/images/...,[],Verizon,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '11.6 x 6.9 x 3.1 inche...,B004E2Z88O,8b781665-d968-4aaf-ac18-f7e353de0de9,Motorola Droid X Essentials Combo Pack ['all G...,https://m.media-amazon.com/images/I/51-DXSMlHa...


In [223]:
df.iloc[1]

main_category                                              All Electronics
title                                        Ce-H22B12-S1 4Kx2K Hdmi 4Port
average_rating                                                         5.0
rating_number                                                            1
features                            [UPC: 662774021904, Weight: 0.600 lbs]
description                                           [HDMI In - HDMI Out]
price                                                            96.551146
images                   [{'thumb': 'https://m.media-amazon.com/images/...
videos                                                                  []
store                                                                 SIIG
categories               [Electronics, Television & Video, Accessories,...
details                  {'Product Dimensions': '0.83 x 4.17 x 2.05 inc...
parent_asin                                                     B00YT6XQSE
product_id               

In [224]:
print(df.iloc[1]['processed_description'])

Ce-H22B12-S1 4Kx2K Hdmi 4Port ['HDMI In - HDMI Out'] All Electronics ['UPC: 662774021904', 'Weight: 0.600 lbs'] 96.5511458333 ['Electronics', 'Television & Video', 'Accessories', 'Cables', 'HDMI Cables']


In [225]:
print(df.iloc[1]['image_url'])

https://m.media-amazon.com/images/I/31OIMoOW70L._AC_.jpg


In [226]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2026 entries, 0 to 2025
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   main_category          2000 non-null   object 
 1   title                  2026 non-null   object 
 2   average_rating         2026 non-null   float64
 3   rating_number          2026 non-null   int64  
 4   features               2026 non-null   object 
 5   description            2026 non-null   object 
 6   price                  2026 non-null   float64
 7   images                 2026 non-null   object 
 8   videos                 2026 non-null   object 
 9   store                  2026 non-null   object 
 10  categories             2026 non-null   object 
 11  details                2026 non-null   object 
 12  parent_asin            2026 non-null   object 
 13  product_id             2026 non-null   object 
 14  processed_description  2026 non-null   object 
 15  imag

In [227]:
data_to_embed = df[["processed_description", "image_url", "average_rating", "rating_number", "price", "parent_asin"]]

In [228]:
data_to_embed

,processed_description,image_url,average_rating,rating_number,price,parent_asin
0,FS-1051 FATSHARK TELEPORTER V3 HEADSET ['Telep...,https://m.media-amazon.com/images/I/41qrX56lsY...,3.5,6,96.551146,B00MCW7G9M
1,Ce-H22B12-S1 4Kx2K Hdmi 4Port ['HDMI In - HDMI...,https://m.media-amazon.com/images/I/31OIMoOW70...,5.0,1,96.551146,B00YT6XQSE
2,Digi-Tatoo Decal Skin Compatible With MacBook ...,https://m.media-amazon.com/images/I/31t4bj9t88...,4.5,246,19.990000,B07SM135LS
3,NotoCity Compatible with Vivoactive 4 band 22m...,https://m.media-amazon.com/images/I/41j56fjX6S...,4.5,233,9.990000,B089CNGZCW
4,Motorola Droid X Essentials Combo Pack ['all G...,https://m.media-amazon.com/images/I/51-DXSMlHa...,3.8,64,14.990000,B004E2Z88O
...,...,...,...,...,...,...
2021,"(2 Packs) 32mm 1/4"" to 3/8"" inch Male to Male ...",https://m.media-amazon.com/images/I/31ARI5oc4j...,3.6,7,96.551146,B00OPYMI2A
2022,i-BLASON Auto Sleep/Wake Google Nexus 10 inch ...,https://m.media-amazon.com/images/I/51C5pgZDJA...,3.8,157,96.551146,B00A1PM340
2023,Charger for Lenovo Laptop Computer 65W 45W USB...,https://m.media-amazon.com/images/I/41hZ69SRkT...,4.6,350,19.320000,B08L4YB3QC
2024,"Samsung XE550C22 - 12.1"" Series 5 Chromebook, ...",https://m.media-amazon.com/images/I/41ZuLno8a0...,2.3,4,96.551146,B07BHQPXXZ


In [229]:
text_to_embed = data_to_embed["processed_description"].tolist()

In [230]:
text_to_embed

["FS-1051 FATSHARK TELEPORTER V3 HEADSET ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'] All Electronics [] 96.5511458333 ['Electronics', 'Television & Video', 'Video Glasses']",
 "Ce-H22B12-S1 4Kx2K Hdmi 4Port ['HDMI In - HDMI Out'] All Electronics ['UPC: 662774021904', 'Weight: 0.600 lbs'] 96.5511458333 ['Electronics', 'Television & Video', 'Accessories', 'Cables', 'HDMI Cables']"

In [231]:
embeddings = get_embeddings_batch(text_to_embed, batch_size=100)

Processed batch 1 / 21
Processed batch 2 / 21
Processed batch 3 / 21
Processed batch 4 / 21
Processed batch 5 / 21
Processed batch 6 / 21
Processed batch 7 / 21
Processed batch 8 / 21
Processed batch 9 / 21
Processed batch 10 / 21
Processed batch 11 / 21
Processed batch 12 / 21
Processed batch 13 / 21
Processed batch 14 / 21
Processed batch 15 / 21
Processed batch 16 / 21
Processed batch 17 / 21
Processed batch 18 / 21
Processed batch 19 / 21
Processed batch 20 / 21
Processed batch 21 / 21


In [232]:
len(embeddings)

2026

In [233]:
pointstructs = []
i = 1
for embedding, (_, data) in zip(embeddings, data_to_embed.iterrows()):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data['processed_description'],
                    model="qdrant/bm25",
                )
            },
            payload=data.to_dict()
        )
    )
    if i % 100 == 0:
        print(f"Prepared {i} / {len(embeddings)} points")
    i += 1

Prepared 100 / 2026 points
Prepared 200 / 2026 points
Prepared 300 / 2026 points
Prepared 400 / 2026 points
Prepared 500 / 2026 points
Prepared 600 / 2026 points
Prepared 700 / 2026 points
Prepared 800 / 2026 points
Prepared 900 / 2026 points
Prepared 1000 / 2026 points
Prepared 1100 / 2026 points
Prepared 1200 / 2026 points
Prepared 1300 / 2026 points
Prepared 1400 / 2026 points
Prepared 1500 / 2026 points
Prepared 1600 / 2026 points
Prepared 1700 / 2026 points
Prepared 1800 / 2026 points
Prepared 1900 / 2026 points
Prepared 2000 / 2026 points


In [234]:
print(pointstructs[0])

id=1 vector={'text-embedding-3-small': [-0.015716552734375, -0.015045166015625, 0.0202789306640625, -0.033233642578125, -0.007572174072265625, -0.032806396484375, 0.041656494140625, 0.051666259765625, 0.03631591796875, -0.0052947998046875, 0.02001953125, 0.0016756057739257812, -0.026275634765625, -0.0101470947265625, 0.00656890869140625, -0.02655029296875, -0.004787445068359375, -0.03009033203125, -0.040863037109375, 0.030914306640625, 0.0357666015625, -0.01226806640625, 0.00150299072265625, -0.0082855224609375, -0.017059326171875, -0.07415771484375, 0.0093841552734375, 0.033538818359375, 0.002765655517578125, 0.0019779205322265625, -0.0027637481689453125, -0.038299560546875, 0.024169921875, -0.0018949508666992188, -0.0374755859375, -0.0252532958984375, -0.00426483154296875, 0.0028171539306640625, -0.047760009765625, -0.0022258758544921875, 0.06573486328125, -0.0247344970703125, -0.0013103485107421875, -0.0049285888671875, 0.016265869140625, -0.0183258056640625, 0.027923583984375, 0.00

In [235]:
batch_size = 128

for start in range(0, len(pointstructs), batch_size):
    batch = pointstructs[start:start + batch_size]
    qdrant_client.upsert(
        collection_name="Amazon_Electronics_Products",
        wait=True,
        points=batch,
    )

print(f"Upserted {len(pointstructs)} points in batches of {batch_size}.")

Upserted 2026 points in batches of 128.


Hybrid Retrieval

In [236]:
def retrieve_data(query, qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Products",
        prefetch = [
            Prefetch(
            query = query_embedding,
            using= "text-embedding-3-small",
            limit = 20
           ),
           Prefetch(
            query = Document(text=query, model="qdrant/bm25"),
            using= "bm25",
            limit = 20
           )
        ],
        query=FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []
    retrieved_context_prices = []
    retrieved_context_images = []
    retrieved_context_rating_numbers = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_contexts.append(result.payload['processed_description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
        retrieved_context_prices.append(result.payload['price'])
        retrieved_context_images.append(result.payload['image_url'])
        retrieved_context_rating_numbers.append(result.payload['rating_number'])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
        "retrieved_context_prices": retrieved_context_prices,
        "retrieved_context_images": retrieved_context_images,
        "retrieved_context_rating_numbers": retrieved_context_rating_numbers
    }

In [237]:
results = retrieve_data("Looking for a good quality wireless mouse with long battery life", qdrant_client, top_k=5)

In [238]:
results

{'retrieved_context_ids': ['B00113ZBYA',
  'B07R32PZZ6',
  'B0BWYJJLSF',
  'B071NFDHX3',
  'B00EYIRKE8'],
 'retrieved_contexts': ['Logitech V200 Cordless Mouse - Silver (931379-0403) [\'From the Manufacturer\', \'Maximize your productivity, where ever you are\\x97at your desk, on the road, or even in the conference room down the hall.\', \'The Logitech V200 Cordless Notebook Mouse is designed for the demanding mobile professional. Compact, with an unsurpassed battery life of up to one year, the V200 guarantees comfort and reliable performance on the road.\', "Its 2.4GHz digital cordless technology provides longer range compared to traditional wireless, and delivers interference-free operation. The 2.4GHz micro-receiver snaps into the mouse for easy transport and turns it off for power efficiency. Invisible Light optical tracking delivers smooth, precise control and the Tilt Wheel Plus Zoom makes side-to-side scrolling effortless. All of this is wrapped up in a durable design that\'s id

In [239]:
results["retrieved_contexts"][0]

'Logitech V200 Cordless Mouse - Silver (931379-0403) [\'From the Manufacturer\', \'Maximize your productivity, where ever you are\\x97at your desk, on the road, or even in the conference room down the hall.\', \'The Logitech V200 Cordless Notebook Mouse is designed for the demanding mobile professional. Compact, with an unsurpassed battery life of up to one year, the V200 guarantees comfort and reliable performance on the road.\', "Its 2.4GHz digital cordless technology provides longer range compared to traditional wireless, and delivers interference-free operation. The 2.4GHz micro-receiver snaps into the mouse for easy transport and turns it off for power efficiency. Invisible Light optical tracking delivers smooth, precise control and the Tilt Wheel Plus Zoom makes side-to-side scrolling effortless. All of this is wrapped up in a durable design that\'s ideal for business-class mobility.", \'Tilt Wheel Plus Zoom. Scroll side-to-side and zoom in on page details.\', \'Tilt Wheel Plus Z